We are going to populate the SQLITE database with fields extract from LLM

In [ ]:
import sqlite3
from typing import Optional
from pydantic import BaseModel
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI()

# Pydantic Model for Structured Output
class Employee(BaseModel):
    full_name: str
    date_of_birth: str
    current_job_title: str
    location: str
    current_salary: int
    hire_date: str
    years_at_company: int
    latest_performance_rating: Optional[float] = None
    total_bonus_last_year: Optional[int] = None
    number_of_promotions: int
    awards_count: int
class Product(BaseModel):
    product_name: str
    summary: str
    target_market: str  # e.g., "B2B", "B2C", "B2B and B2C"
    feature_count: int
    pricing_tiers_count: int
    lowest_tier_price: Optional[int] = None  # None for custom pricing
    highest_tier_price: Optional[int] = None
    roadmap_items_count: int
    latest_roadmap_quarter: Optional[str] = None  # e.g., "Q1 2025"
class Contract(BaseModel):
    contract_number: str
    contract_date: str  # YYYY-MM-DD format
    customer_name: str
    product_name: str  # e.g., "Carllm", "Homellm", "Lifellm"
    tier: str  # e.g., "Enterprise", "Standard", "Custom"
    monthly_fee: int
    duration_months: int
    total_contract_value: int
    features_count: int
    support_level: str  # e.g., "24/7 Enterprise", "Standard", "Dedicated"
    auto_renews: bool
    renewal_notice_days: Optional[int] = None

DB_NAME = "sqldb.db"
# Database Setup
def create_database():
    """Create SQLite database with single employees table"""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS employees (
            employee_id INTEGER PRIMARY KEY AUTOINCREMENT,
            full_name TEXT,
            date_of_birth TEXT,
            current_job_title TEXT,
            location TEXT,
            current_salary INTEGER,
            hire_date TEXT,
            years_at_company INTEGER,
            latest_performance_rating REAL,
            total_bonus_last_year INTEGER,
            number_of_promotions INTEGER,
            awards_count INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    conn.commit()
    conn.close()
    print("Database created successfully!")

# Database Setup for Products
def create_products_table():
    """Add products table to existing database"""
    conn = sqlite3.connect('sqldb.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS products (
            product_id INTEGER PRIMARY KEY AUTOINCREMENT,
            product_name TEXT,
            summary TEXT,
            target_market TEXT,
            feature_count INTEGER,
            pricing_tiers_count INTEGER,
            lowest_tier_price INTEGER,
            highest_tier_price INTEGER,
            roadmap_items_count INTEGER,
            latest_roadmap_quarter TEXT,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    conn.commit()
    conn.close()
    print("Products table created successfully!")

def create_contracts_table():
    """Add contracts table to existing database"""
    conn = sqlite3.connect('sqldb.db')
    cursor = conn.cursor()
    
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS contracts (
            contract_id INTEGER PRIMARY KEY AUTOINCREMENT,
            contract_number TEXT,
            contract_date TEXT,
            customer_name TEXT,
            product_name TEXT,
            tier TEXT,
            monthly_fee INTEGER,
            duration_months INTEGER,
            total_contract_value INTEGER,
            features_count INTEGER,
            support_level TEXT,
            auto_renews BOOLEAN,
            renewal_notice_days INTEGER,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    conn.commit()
    conn.close()
    print("Contracts table created successfully!")


In [ ]:
# LLM Parsing Function
def parse_employee_document(document_text: str) -> Employee:
    """Use OpenAI to parse employee document into structured data"""
    
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {
                "role": "system",
                "content": """You are an HR document parser. Extract key employee information from the provided HR document. If a field cannot be found, simply return "N/A"

Instructions:
- For dates, use YYYY-MM-DD format when possible. If only month/year given, use YYYY-MM-01.
- Calculate years_at_company from hire_date to current date (2024)
- latest_performance_rating should be the most recent year's rating
- total_bonus_last_year should be the most recent year's bonus amount
- number_of_promotions: count explicit promotions mentioned
- awards_count: count total number of awards/recognitions mentioned
"""
            },
            {
                "role": "user",
                "content": f"Parse this employee document:\n\n{document_text}"
            }
        ],
        response_format=Employee,
    )
    
    return completion.choices[0].message.parsed

# Database Insertion Function
def insert_employee(employee: Employee):
    """Insert parsed employee data into SQLite database"""
    conn = sqlite3.connect('sqldb.db')
    cursor = conn.cursor()
    
    try:
        cursor.execute('''
            INSERT INTO employees (
                full_name, date_of_birth, current_job_title, location, 
                current_salary, hire_date, years_at_company, 
                latest_performance_rating, total_bonus_last_year, 
                number_of_promotions, awards_count
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            employee.full_name,
            employee.date_of_birth,
            employee.current_job_title,
            employee.location,
            employee.current_salary,
            employee.hire_date,
            employee.years_at_company,
            employee.latest_performance_rating,
            employee.total_bonus_last_year,
            employee.number_of_promotions,
            employee.awards_count
        ))
        
        employee_id = cursor.lastrowid
        conn.commit()
        print(f"✓ Successfully inserted: {employee.full_name} (ID: {employee_id})")
        return employee_id
        
    except Exception as e:
        conn.rollback()
        print(f"✗ Error inserting employee: {e}")
        raise
    finally:
        conn.close()

# Main Processing Function
def process_employee_document(document_text: str):
    """Complete pipeline: parse document and insert into database"""
    print("Parsing document with LLM...")
    employee = parse_employee_document(document_text)
    
    print(f"\nParsed: {employee.full_name}")
    print(f"  Title: {employee.current_job_title}")
    print(f"  Salary: ${employee.current_salary:,}")
    print(f"  Performance: {employee.latest_performance_rating}/5")
    print(f"  Promotions: {employee.number_of_promotions}")
    
    print("\nInserting into database...")
    employee_id = insert_employee(employee)
    
    return employee_id



In [ ]:
# LLM Parsing Function
def parse_product_document(document_text: str) -> Product:
    """Use OpenAI to parse product document into structured data"""
    
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {
                "role": "system",
                "content": """You are a product document parser. Extract key product metrics from the provided document. If a field cannot be found, simply return "N/A"

Instructions:
- Extract the product name and summary
- Count the total number of features listed
- Count the total number of pricing tiers
- Extract the lowest monthly price (as integer, e.g., 3500 for $3,500/month)
- Extract the highest monthly price (use None if top tier is "custom pricing")
- Count the total number of roadmap items
- Extract the latest/furthest roadmap quarter (e.g., "Q3 2027")
- Determine target_market from the summary: "B2B", "B2C", or "B2B and B2C"
"""
            },
            {
                "role": "user",
                "content": f"Parse this product document:\n\n{document_text}"
            }
        ],
        response_format=Product,
    )
    
    return completion.choices[0].message.parsed

# Database Insertion Function
def insert_product(product: Product):
    """Insert parsed product data into SQLite database"""
    conn = sqlite3.connect('sqldb.db')
    cursor = conn.cursor()
    
    try:
        cursor.execute('''
            INSERT INTO products (
                product_name, summary, target_market, feature_count,
                pricing_tiers_count, lowest_tier_price, highest_tier_price,
                roadmap_items_count, latest_roadmap_quarter
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            product.product_name,
            product.summary,
            product.target_market,
            product.feature_count,
            product.pricing_tiers_count,
            product.lowest_tier_price,
            product.highest_tier_price,
            product.roadmap_items_count,
            product.latest_roadmap_quarter
        ))
        
        product_id = cursor.lastrowid
        conn.commit()
        print(f"✓ Successfully inserted: {product.product_name} (ID: {product_id})")
        return product_id
        
    except Exception as e:
        conn.rollback()
        print(f"✗ Error inserting product: {e}")
        raise
    finally:
        conn.close()

# Main Processing Function
def process_product_document(document_text: str):
    """Complete pipeline: parse document and insert into database"""
    print("Parsing product document with LLM...")
    product = parse_product_document(document_text)
    
    print(f"\nParsed: {product.product_name}")
    print(f"  Target Market: {product.target_market}")
    print(f"  Features: {product.feature_count}")
    # ✅ Fix: only format numerically if it's a number
    low = f"${product.lowest_tier_price:,}" if product.lowest_tier_price is not None else "N/A"
    high = f"${product.highest_tier_price:,}" if isinstance(product.highest_tier_price, (int, float)) else "Custom"

    print(f"  Price Range: {low} - {high}")
    print(f"  Roadmap through: {product.latest_roadmap_quarter}")
    
    print("\nInserting into database...")
    product_id = insert_product(product)
    
    return product_id



In [ ]:
# LLM Parsing Function
def parse_contract_document(document_text: str) -> Contract:
    """Use OpenAI to parse contract document into structured data"""
    
    completion = client.beta.chat.completions.parse(
        model="gpt-4o-2024-08-06",
        messages=[
            {
                "role": "system",
                "content": """You are a contract document parser. Extract key contract metrics from the provided document. If a field cannot be found, simply return "N/A".

Instructions:
- Extract contract_number from the document
- Extract contract_date in YYYY-MM-DD format
- Extract customer_name (the company purchasing, not Insurellm)
- Extract product_name (e.g., "Carllm", "Homellm", "Lifellm", "Claimllm")
- Extract tier from the payment section (e.g., "Enterprise", "Standard", "Core", "Custom")
- Calculate monthly_fee (if variable pricing over contract, use average monthly fee as integer)
- Extract duration_months from the duration/term section
- Calculate total_contract_value (sum of all payments over contract term)
- Count features_count (number of distinct features listed in Features section)
- Determine support_level (e.g., "24/7 Enterprise", "Standard", "Dedicated", based on support description)
- Determine auto_renews (true if contract automatically renews, false otherwise)
- Extract renewal_notice_days (number of days notice required for renewal/termination)
"""
            },
            {
                "role": "user",
                "content": f"Parse this contract document:\n\n{document_text}"
            }
        ],
        response_format=Contract,
    )
    
    return completion.choices[0].message.parsed

# Database Insertion Function
def insert_contract(contract: Contract):
    """Insert parsed contract data into SQLite database"""
    conn = sqlite3.connect('sqldb.db')
    cursor = conn.cursor()
    
    try:
        cursor.execute('''
            INSERT INTO contracts (
                contract_number, contract_date, customer_name, product_name,
                tier, monthly_fee, duration_months, total_contract_value,
                features_count, support_level, auto_renews, renewal_notice_days
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            contract.contract_number,
            contract.contract_date,
            contract.customer_name,
            contract.product_name,
            contract.tier,
            contract.monthly_fee,
            contract.duration_months,
            contract.total_contract_value,
            contract.features_count,
            contract.support_level,
            contract.auto_renews,
            contract.renewal_notice_days
        ))
        
        contract_id = cursor.lastrowid
        conn.commit()
        print(f"✓ Successfully inserted: {contract.customer_name} - {contract.product_name} (ID: {contract_id})")
        return contract_id
        
    except Exception as e:
        conn.rollback()
        print(f"✗ Error inserting contract: {e}")
        raise
    finally:
        conn.close()

# Main Processing Function
def process_contract_document(document_text: str):
    """Complete pipeline: parse document and insert into database"""
    print("Parsing contract document with LLM...")
    contract = parse_contract_document(document_text)
    
    print(f"\nParsed: {contract.contract_number}")
    print(f"  Customer: {contract.customer_name}")
    print(f"  Product: {contract.product_name} ({contract.tier})")
    print(f"  Monthly Fee: ${contract.monthly_fee:,}")
    print(f"  Total Value: ${contract.total_contract_value:,}")
    print(f"  Duration: {contract.duration_months} months")
    print(f"  Auto-renews: {contract.auto_renews}")
    
    print("\nInserting into database...")
    contract_id = insert_contract(contract)
    
    return contract_id



In [ ]:
create_database()
create_products_table()
create_contracts_table()
create_contracts_table()


In [ ]:
#Parse all documents
import os

def process_all_documents():
    base_dir = "knowledge-base"
    for folder in os.listdir(base_dir):
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path):
            continue

        # Decide which processor to use
        if folder == "contracts":
            processor = process_contract_document
        elif folder == "employees":
            processor = process_employee_document
        elif folder == "products":
            processor = process_product_document
        else:
            processor = None

        if processor:
            for filename in os.listdir(folder_path):
                file_path = os.path.join(folder_path, filename)
                if not os.path.isfile(file_path):
                    continue
                try:
                    with open(file_path, "r", encoding="utf-8") as f:
                        doc_text = f.read()
                    print(f"\nProcessing {file_path} ...")
                    processor(doc_text)
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")

process_all_documents()
# 3 min

In [ ]:
#Example
employee_document = """
    # HR Record

    # Alex Harper

    ## Summary
    - **Date of Birth**: March 15, 1993
    - **Job Title**: Sales Development Representative (SDR)
    - **Location**: Denver, Colorado
    - **Current Salary**: $75,000  

    ## Insurellm Career Progression
    - **July 2021**: Joined Insurellm as a Sales Development Representative, focusing on lead generation and nurturing B2B relationships.  
    - **January 2022**: Promoted to Senior Sales Development Representative due to exceptional performance in converting leads into clients.  
    - **October 2022**: Completed an Internal Leadership Training Program, enhancing skills in team collaboration and strategic selling. Currently mentoring junior SDRs.  
    - **April 2023**: Became involved in a cross-departmental project to streamline the customer onboarding process, showcasing initiative and leadership.  

    ## Annual Performance History  
    - **2021**:  
    - **Performance Rating**: 4.5/5  
    - **Key Achievements**: Exceeded lead generation targets by 30%. Introduced a new CRM analytics tool resulting in improved tracking of customer interactions.  

    - **2022**:  
    - **Performance Rating**: 4.8/5  
    - **Key Achievements**: Awarded "SDR of the Year" for outstanding contributions. Instrumental in securing 15 new B2B contracts, surpassing targets by 40%.  

    - **2023**:  
    - **Performance Rating**: 4.7/5  
    - **Key Achievements**: Played a key role in the launch of a new product line with a 25% increase in lead-to-conversion rates. Completed advanced sales negotiation training with high marks.  

    ## Compensation History  
    - **2021**:  
    - **Base Salary**: $55,000  
    - **Bonus**: $5,500 (10% of base due to performance)  

    - **2022**:  
    - **Base Salary**: $65,000 (Promotion to Senior SDR)  
    - **Bonus**: $13,000 (20% of base due to performance)  

    - **2023**:  
    - **Base Salary**: $75,000  
    - **Bonus**: $15,000 (20% of base)  

    ## Other HR Notes  
    - **Training Completed**:  
    - CRM Analytics & Data Management Workshop (2021)  
    - Leadership Training Program (2022)  
    - Advanced Sales Negotiation Course (2023)  

    - **Awards**:  
    - Insurellm "SDR of the Year" Award (2022)  
    - Monthly MVP Recognition (3 times in 2023)  
"""


product_document = """
    # Product Summary

    # Healthllm

    ## Summary

    Healthllm is Insurellm's comprehensive health insurance platform that empowers insurance providers to deliver modern, personalized health coverage. By combining advanced AI technology with healthcare data analytics, Healthllm streamlines every aspect of health insurance operations—from plan design and enrollment to claims processing and member engagement. Built for the complexities of the healthcare industry, Healthllm helps insurers reduce costs, improve member outcomes, and navigate the evolving regulatory landscape with confidence.

    ## Features

    ### 1. Intelligent Plan Design
    Healthllm's AI-powered tools help insurers create competitive health plans by analyzing market trends, member demographics, and healthcare utilization patterns. The platform suggests optimal benefit structures, deductibles, and network configurations to maximize both member value and profitability.

    ### 2. Real-Time Eligibility Verification
    Integrated eligibility checking ensures instant verification of coverage status, reducing claim denials and improving provider satisfaction. The system connects with healthcare provider networks for seamless real-time validation.

    ### 3. AI-Driven Claims Adjudication
    Automated claims processing uses machine learning to review claims for accuracy, identify potential fraud, and expedite legitimate payments. The system learns from historical data to continuously improve processing accuracy and efficiency.

    ### 4. Predictive Healthcare Analytics
    Advanced analytics identify high-risk members who would benefit from preventive care interventions, enabling insurers to implement proactive wellness programs that reduce long-term costs and improve member health outcomes.

    ### 5. Provider Network Management
    Comprehensive tools for managing provider relationships, contract negotiations, and network adequacy. The platform tracks provider performance metrics and identifies opportunities for cost savings through strategic partnerships.

    ### 6. Member Engagement Platform
    A mobile-first member portal provides easy access to coverage information, digital ID cards, claims status, provider directories, and telehealth services. Push notifications keep members informed about preventive care opportunities and wellness programs.

    ### 7. Medication Management
    Integration with pharmacy benefit managers enables formulary management, prior authorization automation, and medication adherence tracking. AI-powered tools identify cost-saving generic alternatives and flag potential drug interactions.

    ### 8. Regulatory Compliance Engine
    Built-in compliance monitoring ensures adherence to ACA requirements, state mandates, and HIPAA regulations. Automated reporting simplifies regulatory filings and reduces compliance risk.

    ## Pricing

    Healthllm offers tiered pricing to serve health insurers across the spectrum:

    - **Essential Tier:** $8,000/month for regional health plans, providing core platform capabilities and standard integrations.
    - **Professional Tier:** $15,000/month for larger insurers, including advanced analytics, predictive modeling, and expanded integration options.
    - **Enterprise Tier:** Custom pricing for national carriers and health systems requiring extensive customization, multi-state support, dedicated infrastructure, and premium support services.

    All plans include implementation support, staff training, and ongoing platform enhancements.

    ## Roadmap

    Healthllm's strategic development roadmap includes:

    - **Q1 2025:** Launch of Healthllm version 1.0 featuring core claims processing, eligibility verification, and member portal.
    - **Q3 2025:** Introduction of predictive analytics module for population health management and risk stratification.
    - **Q1 2026:** Release of advanced AI claims adjudication with automated fraud detection and payment optimization.
    - **Q3 2026:** Launch of social determinants of health (SDOH) integration, enabling holistic member care and targeted interventions.
    - **Q1 2027:** Introduction of value-based care management tools supporting ACO and bundled payment models.
    - **Q3 2027:** Global expansion with international healthcare system integrations and multi-language support.

    Healthllm represents Insurellm's commitment to transforming health insurance through technology that improves outcomes for insurers, providers, and members alike. Join us in building the future of health insurance!
"""

contract_document = """
    # Contract with Velocity Auto Solutions for Carllm

    **Contract Date:** October 1, 2023  
    **Contract Number:** C-12345-2023  
    **Client:** Velocity Auto Solutions  
    **Product:** Carllm Auto Insurance Solution  

    ---

    ## Terms

    1. **Duration**: This contract is effective for a period of 12 months from the contract date.  
    2. **Payment Schedule**: Velocity Auto Solutions agrees to pay Insurellm the total fee associated with the selected subscription tier on a monthly basis, beginning on the contract date.  
    3. **Confidentiality**: Both parties agree to keep all proprietary information confidential and not to disclose it to any third parties without written consent.  
    4. **Intellectual Property**: All components of Carllm and any related technology are the property of Insurellm, and license is granted to Velocity Auto Solutions for internal use only.  

    ## Renewal

    1. **Automatic Renewal**: This contract will automatically renew for successive 12-month periods unless either party provides written notice at least 30 days prior to the end of the initial term or any renewal term.  
    2. **Rate Adjustment**: Subscription pricing may be subject to adjustment, with Insurellm providing a 60-day advance notice of any changes prior to renewal.  

    ## Features

    1. **Included Features**:  
    - AI-Powered Risk Assessment  
    - Instant Quoting and Customizable Coverage Plans  
    - Fraud Detection Systems  
    - Customer Insights Dashboard  
    - Automated Customer Support  

    2. **Feature Enhancements**: Velocity Auto Solutions will receive updates to the Carllm product as outlined in the Insurellm 2025-2026 Roadmap, including mobile integration and telematics-based pricing enhancements.

    ## Support

    1. **Customer Support**: Velocity Auto Solutions will have access to Insurellm’s customer support team via email or chatbot, available 24/7.  
    2. **Technical Maintenance**: Regular maintenance and updates to the Carllm platform will be conducted by Insurellm, with any downtime communicated in advance.  
    3. **Training & Resources**: Initial training sessions will be provided for Velocity Auto Solutions’ staff to ensure effective use of the Carllm suite. Regular resources and documentation will be made available online.

    ---

    **Accepted and Agreed:**  
    **For Velocity Auto Solutions**  
    Signature: _____________________  
    Name: John Doe  
    Title: CEO  
    Date: _____________________  

    **For Insurellm**  
    Signature: _____________________  
    Name: Jane Smith  
    Title: VP of Sales  
    Date: _____________________
"""

process_employee_document(employee_document)
process_product_document(product_document)
process_contract_document(contract_document)